# Orbitopia — NASA data check (run in Google Colab)
This notebook **re-fetches every NASA number used in Orbitopia** so judges and teammates can verify it.
1. DONKI: the 5 real Sun weeks (biggest flare per day, SEP detected?, fastest CME) and the in-game storm-power rule
2. RadLab: ISS DosTel1 dose rate, quiet day vs storm day
3. OSDR: the study IDs used on the fact cards
4. (Optional) Export a fresh DONKI snapshot for `js/data.js`

*Code drafted with AI assistance (Claude, Anthropic) and reviewed by the team.*

In [ ]:
import requests, pandas as pd, json, datetime as dt
DONKI = "https://ccmc.gsfc.nasa.gov/DONKI/WS/get/"   # official DONKI URL since Sept 2026 (was kauai.ccmc.gsfc.nasa.gov)
WEEKS = {"may2024":("2024-05-08","2024-05-14"), "oct2024":("2024-10-03","2024-10-09"),
         "sep2017":("2017-09-04","2017-09-10"), "nov2025":("2025-11-05","2025-11-11"), "nov2026":("2026-11-05","2026-11-11")}
def get(kind, a, b, **kw):
    r = requests.get(DONKI + kind, params={"startDate": a, "endDate": b, **kw}, timeout=60); r.raise_for_status()
    return r.json() if r.text.strip() else []

In [ ]:
RANK = {"A":1,"B":2,"C":3,"M":4,"X":5}
def rank(c): return RANK.get(c[0],0)*100 + float(c[1:] or 0) if c else 0
def storm_level(flare, sep):
    # Same rule as js/engine.js
    if not sep: return 0
    if not flare: return 1
    L, m = flare[0], float(flare[1:])
    if L == "X": return 4 if m >= 5 else 3 if m >= 2 else 2
    if L == "M": return 2 if m >= 5 else 1
    return 1
rows = []
for wid, (a, b) in WEEKS.items():
    fl = get("FLR", a, b); sep = get("SEP", a, b)
    cme = get("CMEAnalysis", a, b, mostAccurateOnly="true", speed=0, halfAngle=0)
    for d in pd.date_range(a, b):
        ds = d.strftime("%Y-%m-%d")
        f = [x for x in fl if (x.get("peakTime") or "").startswith(ds)]
        best = max(f, key=lambda x: rank(x["classType"]), default=None)
        s = [x for x in sep if x["eventTime"].startswith(ds)]
        c = [x["speed"] for x in cme if (x.get("time21_5") or "").startswith(ds) and x.get("speed")]
        rows.append(dict(week=wid, date=ds, biggest_flare=best and best["classType"], peak=best and best["peakTime"][11:16],
                         n_flares=len(f), sep_detected=bool(s), first_sep=s[0]["eventTime"][11:16] if s else None,
                         fastest_cme_kms=max(c) if c else None, game_storm_power=storm_level(best and best["classType"], bool(s))))
df = pd.DataFrame(rows); df

Compare `game_storm_power` with the app's **Real NASA Science → The real Sun weeks** table. The game's archive keeps the biggest flare of each day; for the older weeks only flares of class M5 or above were kept, so days with only small flares show as calm.

In [ ]:
RAD = "https://visualization.osdr.nasa.gov/radlab/api/"
def radlab(t0, t1, inst="DosTel1"):
    q = f"instrument_id = {inst} AND timestamp >= {t0} AND timestamp < {t1}"
    r = requests.get(RAD, params={"query": q, "fields+": "absorbed_dose_rate,latitude,longitude", "format": "json"}, timeout=180)
    r.raise_for_status(); return pd.DataFrame(r.json())
quiet = radlab("2024-04-20T00:00", "2024-04-20T03:00"); storm = radlab("2024-05-11T00:00", "2024-05-11T03:00")
for name, d in [("quiet 20 Apr 2024", quiet), ("storm 11 May 2024", storm)]:
    v = pd.to_numeric(d["absorbed_dose_rate"], errors="coerce")
    print(f"{name}: n={len(v)} min={v.min():.2f} max={v.max():.2f} mean={v.mean():.2f} µGy/h")

In [ ]:
import matplotlib.pyplot as plt
ax = plt.figure(figsize=(9,3)).gca()
for name, d in [("20 Apr 2024 (quiet)", quiet), ("11 May 2024 (storm)", storm)]:
    ax.plot(pd.to_numeric(d["absorbed_dose_rate"], errors="coerce").values, label=name)
ax.set_ylabel("µGy/h"); ax.set_xlabel("sample (~100 s apart)"); ax.legend(); ax.set_title("ISS DosTel1 — NASA OSDR RadLab"); plt.show()
storm["absorbed_dose_rate"] = pd.to_numeric(storm["absorbed_dose_rate"], errors="coerce")
print(storm.sort_values("absorbed_dose_rate", ascending=False).head(3)[["timestamp","absorbed_dose_rate","latitude","longitude"]])

In [ ]:
OSDR = "https://osdr.nasa.gov/osdr/data/search"
for term in ["lettuce International Space Station", "bone loss spaceflight", "muscle atrophy spaceflight", "space radiation mice"]:
    r = requests.get(OSDR, params={"term": term, "type": "cgene", "size": 5}, timeout=60).json()
    print("\n#", term)
    for h in r["hits"]["hits"]:
        s = h["_source"]; print(" ", s.get("Accession"), "-", (s.get("Study Title") or "")[:100])

In [ ]:
# Optional: fresh 30-day snapshot to paste into js/data.js (window.ORB_DATA.snapshot)
end = dt.date.today(); start = end - dt.timedelta(days=30)
fl = get("FLR", start.isoformat(), end.isoformat()); sep = get("SEP", start.isoformat(), end.isoformat())
snap = {"fetched": end.isoformat(), "window": f"{start} to {end}", "flares": [[f["peakTime"][:10], f["classType"]] for f in fl], "sepCount": len(sep)}
print(json.dumps(snap))